In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# 1. Load the data
# stop_level_diagnostic.csv contains historical reliability (SSRS) for each route_stop_id

PATH = "<PROCESSED_DATA_DIR>/"
df_ssrs = pd.read_csv(PATH + 'stop_level_diagnostic.csv')

# trip_stops_long.csv contains execution records for each trip
df_events = pd.read_csv(PATH + 'trip_stop_events.csv')

# 2. Filter out trips without a route_id
# We ensure the event record has a valid route_id (removes corrupted or manual trip entries)
df_events_filtered = df_events.dropna(subset=['route_id'])

# 3. Merge SSRS scores into the filtered trip events
# Use a LEFT JOIN on 'route_stop_id' to assign historical scores to each planned stop
df_merged = df_events_filtered.merge(
    df_ssrs[['route_stop_id', 'ssrs']], 
    on='route_stop_id', 
    how='left'
)

# 4. Final selection and Output
# Keep only the requested columns for the next stage of Trip-level aggregation
ssrs_output = df_merged[['trip_id', 'route_stop_id', 'ssrs']]

# Save to a new CSV file
ssrs_output.to_csv(PATH + '/trip_events_with_ssrs.csv', index=False)

# Optional: Print summary to verify the filtering process
print(f"Original records: {len(df_events)}")
print(f"Records after filtering out missing route_id: {len(df_events_filtered)}")
print(f"Final output saved with {len(ssrs_output)} rows.")
print(ssrs_output.head())

# Merge with timing outlier score

In [ ]:
# 1. Load the merged event table with SSRS (from your previous step)
# This file contains trip_id, route_stop_id, and ssrs
df_main = pd.read_csv(PATH + 'trip_events_with_ssrs.csv')

# 2. Load the outlier detection results
# This file contains trip_id, route_stop_id, and various outlier metrics (LOF score, etc.)
df_outliers = pd.read_csv(PATH + 'trip_stop_outliers.csv')

# 3. Merge the two files completely based on both trip_id and route_stop_id
# We use a LEFT JOIN to keep all records from our main events table.
# This ensures that even if a stop didn't produce an outlier score, the record remains.
df_combined_full = df_main.merge(
    df_outliers, 
    on=['trip_id', 'route_stop_id'], 
    how='left'
)

# 4. Save the full integrated dataset
# This file will be the foundation for Phase 3 (SQ1 alignment)
df_combined_full.to_csv(PATH + '/trip_stop_diagnostic_integrated.csv', index=False)

# Summary of the merge
print(f"Merge Complete!")
print(f"Total records in main table: {len(df_main)}")
print(f"Total columns in final integrated table: {len(df_combined_full.columns)}")
print(df_combined_full.head())

In [ ]:
# If outlier is NULL -> score = ssrs
# If outlier is NOT NULL -> score = 0.5 * ssrs + 0.5 * outlier
df = pd.read_csv(PATH + 'trip_stop_diagnostic_integrated.csv')
df['stop_level_score'] = np.where(
    df['final_outlier_score'].isna(),
    df['ssrs'], # Case: NULL
    (0.5 * df['ssrs']) + (0.5 * df['final_outlier_score']) # Case: NOT NULL
)

print(df[['route_stop_id', 'ssrs', 'final_outlier_score', 'stop_level_score']].head())

df.to_csv(PATH + '/trip_stop_final_score.csv', index=False)

In [ ]:
# Group by trip_id and calculate the mean ssrs for each trip
trip_ssrs = df_merged.groupby('trip_id')['ssrs'].mean().reset_index()

# Rename the column for clarity
trip_ssrs.columns = ['trip_id', 'avg_ssrs']

# Preview the results
print(trip_ssrs.head())

In [ ]:
# Calculate descriptive statistics for the 'ssrs' column in df_ssrs
print(trip_ssrs['avg_ssrs'].describe())

In [ ]:
route_stop_ssrs = df_merged.groupby('route_stop_id')['ssrs'].mean().reset_index()
route_stop_ssrs.columns = ['route_stop_id', 'avg_ssrs']

zero_route_stops = route_stop_ssrs[route_stop_ssrs['avg_ssrs'] == 0]['route_stop_id']
zero_count = len(zero_route_stops)
zero_ids = zero_route_stops.tolist()

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("white")
plt.figure(figsize=(10, 6))

sns.histplot(route_stop_ssrs['avg_ssrs'], bins=20, kde=True, color='#7a0118ff', alpha=0.8)

plt.axvline(route_stop_ssrs['avg_ssrs'].mean(), color='#FFD700', linestyle='--', label=f'Mean: {route_stop_ssrs["avg_ssrs"].mean():.2f}')
plt.axvline(route_stop_ssrs['avg_ssrs'].median(), color='lightgreen', linestyle='-', label=f'Median: {route_stop_ssrs["avg_ssrs"].median():.2f}')

if zero_count > 0:
    plt.axvline(0, color='red', linestyle=':', label=f'Score=0 route_stop_id count: {zero_count}')
    plt.text(
        0.05,
        plt.ylim()[1] * 0.20,
        f'Stop ID that got 0 Score: {zero_count}\nIDs (first 10): {zero_ids[:10]}',
        color='red',
        fontsize=12,
        bbox=dict(facecolor='white', alpha=0.8, edgecolor='red')
    )

plt.title('Distribution of Stop-Level Missing Scores', fontsize=20)
plt.xlabel('Missing Score', fontsize=15)
plt.ylabel('Number of Route Stops', fontsize=15)
plt.legend(loc='upper left', fontsize=12)

plt.grid(False)
plt.show()

In [ ]:
# 1. Aggregate Stop-level scores to Trip-level
# We use groupby('trip_id') to calculate the mean 'stop_level_score' for each trip.
# This represents the average execution quality of all stops within that specific trip.
df_integrated = pd.read_csv(PATH + 'trip_stop_final_score.csv')
trip_level_scores = df_integrated.groupby('trip_id')['stop_level_score'].mean().reset_index()

# 2. Rename columns for clarity
# Renaming 'stop_level_score' to 'trip_performance_avg_score' to indicate it's now an aggregated metric.
trip_level_scores.columns = ['trip_id', 'trip_final_trip_score(missing+outlier)']

# 3. Data Granularity Transition Summary
# This step converts the data from Stop-level (~82,000 rows) to Trip-level (~8,700 rows).
# Each row now represents the overall "Confidence" of a single vehicle run.
print(f"Aggregation complete: Transformed {len(df_integrated)} stop records into {len(trip_level_scores)} trip scores.")

# 4. Preview the results
print(trip_level_scores.head())

trip_level_scores.to_csv(PATH + '/trip_score_missing_outlier.csv', index=False)

# Merge with final_trip_score(gps+trip_exec)

In [ ]:
# 1. Load the calculated SQ2 Trip-level results
# df_sq2 contains the aggregated scores for missing and outlier data.
df_sq2 = pd.read_csv(PATH + 'trip_score_missing_outlier.csv')

# 2. Load the original trip_score data
# This file typically contains other metrics like gps_health_score, etc.
df_trip_score = pd.read_csv(PATH + 'trip_score.csv')

# 3. Perform the Merge
# Use how='left' to keep df_sq2 as the primary list, 
# or 'outer' to retain all trip_ids from both datasets.
final_report = pd.merge(
    df_sq2, 
    df_trip_score, 
    on='trip_id', 
    how='left',
    suffixes=('', '_drop') # Mark overlapping columns from the right DF for removal
)

# 4. Handle overlapping columns
cols_to_drop = [c for c in final_report.columns if c.endswith('_drop')]
final_report = final_report.drop(columns=cols_to_drop)

# 5. Round the values to 2 decimal places
# We use .round(2) to ensure the numbers are mathematically rounded.
target_col = 'trip_final_trip_score(missing+outlier)'

if target_col in final_report.columns:
    final_report[target_col] = final_report[target_col].round(2)

# 6. Move the specific score column to the last position
other_cols = [c for c in final_report.columns if c != target_col]
final_report = final_report[other_cols + [target_col]]

# 7. Export the results
# When exporting to CSV, pandas will now save the rounded values.
final_report.to_csv(PATH + '/final_trip_score.csv', index=False)

print(f"Success: '{target_col}' rounded to 2 decimal places and moved to the end.")
print(final_report[[target_col]].head())

In [ ]:
# 1. Load the data
# stop_level_diagnostic.csv contains historical reliability (SSRS) for each route_stop_id

PATH = "<PROCESSED_DATA_DIR>/"
df = pd.read_csv(PATH + 'trips_final_confidence_scores.csv')

# trip_stops_long.csv contains execution records for each trip
df_events = pd.read_csv(PATH + 'trip_stop_events.csv')

# 2. Filter out trips without a route_id
# We ensure the event record has a valid route_id (removes corrupted or manual trip entries)

# 3. Merge SSRS scores into the filtered trip events
# Use a LEFT JOIN on 'route_stop_id' to assign historical scores to each planned stop
df_merged = df_events.merge(
    df, 
    on='trip_id', 
    how='left'
)

df_merged.to_csv(PATH + '/trip_events_with_scores.csv', index=False)